# Fraud Detection - Feature Engineering (Python/Pandas)

This notebook performs feature engineering for fraud detection using standard Python libraries instead of PySpark.

## Required Packages

Make sure you have the following packages installed:

```bash
pip install pandas psycopg2-binary sqlalchemy numpy matplotlib seaborn
```

## Database Connection

The notebook connects to PostgreSQL database using:
- **psycopg2**: PostgreSQL adapter for Python
- **sqlalchemy**: SQL toolkit and Object-Relational Mapping (ORM) library  
- **pandas**: Data manipulation and analysis library

## Key Changes from PySpark Version

- ✅ Uses `pandas.read_sql()` instead of Spark DataFrames
- ✅ Direct PostgreSQL connection via `psycopg2` and `sqlalchemy`
- ✅ Memory-efficient processing with chunking support
- ✅ Simpler setup without Spark cluster configuration

In [1]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine
from datetime import datetime, timedelta
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from sqlalchemy import create_engine
import warnings
from datetime import datetime, timedelta

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [3]:
# Database connection parameters
# Note: Run the connection in the terminal first:
# psql -h 127.0.0.1 -p 5432 -U dfstechbi -d db_fraud
from urllib.parse import quote_plus
from sqlalchemy import text

DB_CONFIG = {
    'host': '127.0.0.1',
    'port': '5432',
    'database': 'db_fraud',
    'user': 'dfstechbi',
    'password': 'DfsTeChB1@923'  # You'll enter this when prompted
}

# Prompt for password
from getpass import getpass
DB_CONFIG['password'] = quote_plus(getpass('Enter database password: '))

# Create SQLAlchemy engine
connection_string = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
engine = create_engine(connection_string)

# Test connection
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version();"))
        print("✅ Database connection successful!")
        print(f"PostgreSQL version: {result.fetchone()[0].split(',')[0]}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Database connection successful!
PostgreSQL version: PostgreSQL 17.6 on x86_64-pc-linux-gnu


In [4]:

# Helper function to execute SQL queries and return pandas DataFrames
def fetch_data(query, chunksize=None):
    """
    Execute SQL query and return pandas DataFrame
    
    Args:
        query (str): SQL query to execute
        chunksize (int, optional): If specified, return an iterator of DataFrames
        
    Returns:
        pandas.DataFrame or iterator: Query results
    """
    try:
        if chunksize:
            return pd.read_sql(query, engine, chunksize=chunksize)
        else:
            return pd.read_sql(query, engine)
    except Exception as e:
        print(f"❌ Query execution failed: {str(e)}")
        raise

# Helper function to get table info
def get_table_info(table_name):
    """Get basic information about a table"""
    query = f"""
    SELECT 
        column_name,
        data_type,
        is_nullable,
        column_default
    FROM information_schema.columns 
    WHERE table_name = '{table_name}'
    ORDER BY ordinal_position;
    """
    return fetch_data(query)

# Helper function to get table row count
def get_row_count(table_name):
    """Get row count for a table"""
    query = f"SELECT COUNT(*) as row_count FROM {table_name}"
    result = fetch_data(query)
    return result.iloc[0]['row_count']

print("🔧 Configuration Summary:")
print("   • Using pandas + psycopg2 for data access")
print("   • SQLAlchemy engine with connection pooling")
print("   • Statement timeout: 5 minutes")
print("   • Helper functions available:")
print("     - fetch_data(query, chunksize=None)")
print("     - get_table_info(table_name)")
print("     - get_row_count(table_name)")
print("📚 Ready for data analysis!")

🔧 Configuration Summary:
   • Using pandas + psycopg2 for data access
   • SQLAlchemy engine with connection pooling
   • Statement timeout: 5 minutes
   • Helper functions available:
     - fetch_data(query, chunksize=None)
     - get_table_info(table_name)
     - get_row_count(table_name)
📚 Ready for data analysis!


In [5]:
# Example usage of the database connection

# Example 1: Get list of all tables in the database
print("📋 Getting list of tables...")
tables_query = """
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'public' 
ORDER BY table_name;
"""

try:
    tables_df = fetch_data(tables_query)
    print(f"Found {len(tables_df)} tables:")
    for table in tables_df['table_name'].head(10):  # Show first 10 tables
        print(f"   • {table}")
    if len(tables_df) > 10:
        print(f"   ... and {len(tables_df) - 10} more tables")
except Exception as e:
    print(f"Error fetching tables: {e}")

print("\n" + "="*50)
print("🚀 Ready to start data analysis!")
print("="*50)

📋 Getting list of tables...
Found 10 tables:
   • fraud
   • iar_temp
   • stixor_iar
   • stixor_iar_20250701_sample
   • stixor_iar_jul
   • stixor_iar_jul_fraud_only_with_mbar
   • stixor_iar_jul_non_fraud_only
   • stixor_locations
   • stixor_locations_jul
   • stixor_mbar_v

🚀 Ready to start data analysis!


In [6]:
# Load July customer senders from saved parquet
df_july_customer_senders = pd.read_parquet("../data/july_2025_customer_senders")
print(f"📊 Loaded July customer senders: {len(df_july_customer_senders):,}")

📊 Loaded July customer senders: 22,579,490


In [ ]:
table_name = "public.stixor_iar"

print("🔧 Setting up date range for filtering...")

# Create date range for filtering
start_date = "2025-06-01"
end_date = "2025-07-05"

print(f"📊 Loading data from {table_name} for date range {start_date} to {end_date}")

# Load data using pandas with date filtering
query = f"""
SELECT  ac_from,
        ac_to,
        trans_initiate_time,
        trx_amt,
        trx_channel
FROM {table_name}
WHERE data_date BETWEEN '{start_date}' AND '{end_date}'
"""

print("\n🔗 Loading data from PostgreSQL...")
df = fetch_data(query)

print(f"✅ Loaded {len(df):,} rows from {table_name}")
print(f"📊 Data shape: {df.shape}")


🔧 Setting up date range for filtering...
📊 Loading data from public.stixor_iar for date range 2025-06-01 to 2025-07-05

🔗 Loading data from PostgreSQL...
